In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [17]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl

wanted_issues = [
    "Amalgamated with 12559, then Delete"]

df_issue = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_issue = df_issue[['ISSUE','Network ID', 'Network Name', 'Station ID',  'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

df_issue["old_station_id"] = pd.to_numeric(df_issue["Station ID"], errors="coerce").astype("Int64")


df_issue["new_station_id"] = 12559
df_issue["new_network_id"] = 141

In [ ]:
native_id AFU corresponds to the station 12342 under 5,BCH-GSO-HMP.

In [18]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND m_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_station_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_station_id,

        # n_new
        new_station_id,

        # n_overlap
        old_station_id,
        new_station_id,

        # n_overlap_same_datum
        old_station_id,
        new_station_id,
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = df_issue.apply(
    lambda r: count_station_stats(
        r["old_station_id"],
        r["new_station_id"],
        engine
    ),
    axis=1
)

df_issue[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

In [20]:
df_issue

,ISSUE,Network ID,Network Name,Station ID,Native ID,Unique Names,Unique Location (String),old_station_id,new_station_id,new_network_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,"Amalgamated with 12559, then Delete",9.0,BC-Air -> BCH-SiteC,12213.0,E304453 -> AFU,Peace Valley Attachie Flat Upper Terrace_60 ->...,"121.41944 W, 56.231213 N, Elev. 480 m",12213,12559,141,200949,0,0,0


In [21]:
from sqlalchemy import text

SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :new_station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_MOVE = text("""
WITH updated AS (
    UPDATE obs_raw o
    SET history_id = :new_hist_id
    FROM meta_history h_old
    WHERE o.history_id = h_old.history_id
      AND h_old.station_id = :old_station_id

      AND NOT EXISTS (
          SELECT 1
          FROM obs_raw o_new
          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
          WHERE h_new.station_id = :new_station_id
            AND o_new.obs_time = o.obs_time
            AND o_new.vars_id  = o.vars_id
      )
    RETURNING 1
)
SELECT COUNT(*) FROM updated;
""")

def move_station_obs_fast(
    old_station_id,
    new_station_id,
    engine
):
    with engine.begin() as conn:
        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {"new_station_id": new_station_id}
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_station_id} has no history, skipping.")
            return 0

        n_move = conn.execute(
            SQL_MOVE,
            {
                "old_station_id": old_station_id,
                "new_station_id": new_station_id,
                "new_hist_id": new_hist_id,
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from station {old_station_id} → {new_station_id}"
        )

        return n_move

results = []

for _, row in df_issue.iterrows():
    n = move_station_obs_fast(
        row["old_station_id"],
        row["new_station_id"],
        engine
    )
    results.append(n)

df_issue["n_moved"] = results

Moved 200949 rows from station 12213 → 12559


### Delete old station

In [30]:
old_station_id = 12213
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 12213

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 12213: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
